In [ ]:
import torch
import torch.nn as nn

from mslandcover.models import DeepLabV3Plus, ResNetBackbone, UNet, ResNetBackboneUNet, AttentionUNet
from mslandcover.utils import load_pth

In [2]:
weights_path = './weights/resnet152_202505/hires_simclr_bands3_size256_batch128_randinitfalse/resnet152/hires_simclr.pth'

backbone = ResNetBackboneUNet(in_channels=3)
weights = torch.load(weights_path, map_location='cpu')

# model = DeepLabV3Plus(
#     backbone=ResNetBackbone(
#         arch='resnet152',
#         pretrained=False,
#         out_channels=2048,
#         norm_layer=nn.BatchNorm2d,
#         dilation=True
#     ),
#     num_classes=17,
#     aux_loss=True,
#     norm_layer=nn.BatchNorm2d
# )

/var/folders/1j/_j01624x6y33h54v02qwk81r0000gn/T/ipykernel_48804/3799342074.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(weights_path, map_locati

In [3]:
def adjust_backbone_weights(weights):
    new_weights = {}
    for key in weights.keys():
        if key.startswith('encoder.'):
            new_key = key.replace('encoder.', '')
            
            # initial layers should match up
            if new_key == 'conv1.weight':
                new_key = 'initial.0.weight'
            elif new_key == 'bn1.weight':
                new_key = 'initial.1.weight'
            elif new_key == 'bn1.bias':
                new_key = 'initial.1.bias'
            elif new_key == 'bn1.running_mean':
                new_key = 'initial.1.running_mean'
            elif new_key == 'bn1.running_var':
                new_key = 'initial.1.running_var'
            elif new_key == 'bn1.num_batches_tracked':
                new_key = 'initial.1.num_batches_tracked'
                
            new_weights[new_key] = weights[key]
        
    return new_weights

In [4]:
backbone.load_state_dict(adjust_backbone_weights(weights))

<All keys matched successfully>

In [5]:
model = UNet(
    backbone=backbone,
    num_classes=8,
)

In [6]:
x = torch.randn(2, 3, 256, 256)
model.eval()
with torch.no_grad():
    y = model(x)

x_o: torch.Size([2, 64, 128, 128]), x1: torch.Size([2, 64, 64, 64]), x2: torch.Size([2, 256, 64, 64]), x3: torch.Size([2, 512, 32, 32]), x4: torch.Size([2, 1024, 16, 16]), x5: torch.Size([2, 2048, 8, 8])
torch.Size([2, 2048, 8, 8])
0 torch.Size([2, 2048, 8, 8]) torch.Size([2, 1024, 16, 16])
1 torch.Size([2, 1024, 16, 16]) torch.Size([2, 512, 32, 32])
2 torch.Size([2, 512, 32, 32]) torch.Size([2, 256, 64, 64])
3 torch.Size([2, 256, 64, 64]) torch.Size([2, 64, 128, 128])


In [16]:
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params}')

Total parameters: 107918152


In [7]:
y.shape

torch.Size([2, 8, 128, 128])

In [8]:
import pandas as pd
splits_df = pd.read_csv('./data/splits/splits.csv')

In [9]:
splits_df = splits_df.loc[splits_df['n_train'] == 250]

In [10]:
splits_df.loc[splits_df['fold'] == 1]

,n_train,fold,split_1,split_2,split_3,split_4
0,250,1,train,val,val,val


In [11]:
# select column where entry is 'train'
train_splits = [split for split in splits_df.columns if splits_df[split].iloc[0] == 'train']
val_splits = [split for split in splits_df.columns if splits_df[split].iloc[0] == 'val']
print("Train splits:", train_splits)
print("Validation splits:", val_splits)

Train splits: ['split_1']
Validation splits: ['split_2', 'split_3', 'split_4']


In [12]:
from glob import glob
input_paths = [file for split in train_splits for file in glob(f'./data/splits/{split}/input/*.tif')]
target_paths = [file for split in train_splits for file in glob(f'./data/splits/{split}/target/*.tif')]

In [13]:
print(len(input_paths))

250


In [14]:
splits_df.loc[splits_df['spl

SyntaxError: unterminated string literal (detected at line 1) (1187240453.py, line 1)

In [ ]:
model.backbone.initial.

AttributeError: 'Sequential' object has no attribute 'conv2d'

In [ ]:
model.backbone.initial[1].weight

Parameter containing:
tensor([ 1.8874,  1.8265,  4.1396,  6.2449,  2.4095,  2.0594,  1.9892,  1.9182,
         3.4943,  1.4144,  1.5107,  2.7814, -2.1727,  1.4403,  2.0242,  4.3433,
         4.3720,  1.9215,  1.6825,  2.0796,  1.4970,  1.4553,  2.3145,  1.6794,
         1.7451,  1.6636,  2.3090,  1.9449,  2.3247,  9.4286,  1.4131,  2.8270,
         1.6098,  1.9610,  1.9100,  5.0363,  1.9559,  1.4876,  1.5556,  1.7187,
         3.1604,  1.8487,  1.8618,  1.5315,  9.6446,  1.8686,  2.0230,  1.9094,
         1.3523,  2.1720,  2.2110,  3.0915,  1.5797,  2.0248,  2.4681,  1.5839,
         1.5410,  1.6058,  1.4514,  1.6539,  1.5067,  1.7655,  1.8912,  1.9661],
       requires_grad=True)